# Multi-Seed Oracle-Ceiling Check

**New notebook.** The earlier multi-seed variance check (`multi_seed_variance.ipynb`)
only compared F1 *at one fixed threshold* (`val_p99`) across 3 seeds. It never
checked each seed's own **oracle ceiling** — the best F1 achievable with *any*
threshold for that specific trained model. Since the deployed model's ceiling
(0.686 on `cc1_test`) is a hard mathematical limit no threshold can cross, the
real question for pushing past 0.7 is: **does a different random initialization
produce a model with a genuinely higher ceiling?**

**Design**: reuses the exact validated architecture/hyperparameters
(`hidden1=64, hidden2=32, latent_dim=32, beta_max=0.01`) — only the random seed
varies. This isolates seed-to-seed variance from any architecture change.
Seed 42's model is the existing, already-trained `vae_cc1.pt` (loaded, not
retrained); 4 new seeds are trained fresh.

**Model-selection discipline, stated up front**: the only fully leak-free way
to pick a "best" seed is by lowest validation loss (computed on `cc1_val`,
which is 100% normal — no anomaly information leaks in). All 5 seeds' actual
test-set oracle ceilings are also shown, transparently, so you can see the
full spread — but selecting a seed *because* it happens to score best on
`cc1_test` (rather than by validation loss) would be a mild, 1-in-5 form of
test-set peeking. Both views are reported; which one you use for your final
number is your call, made with the trade-off visible.

In [2]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pickle, os, time
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, precision_score, recall_score, f1_score

BASE      = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
DATA_DIR  = os.path.join(BASE, 'data', 'processed', 'windows_cc1')
MODEL_DIR = os.path.join(BASE, 'models')

DEVICE = torch.device('cpu')
NEW_SEEDS = [7, 123, 2024, 999]   # seed 42 reuses the existing trained model, not retrained

HIDDEN1, HIDDEN2, LATENT_DIM, BETA_MAX = 64, 32, 32, 0.01   # fixed — matches the validated deployed model exactly
WARMUP_EPOCHS, FULL_MAX_EPOCHS, FULL_PATIENCE = 10, 300, 20
LR, BATCH_SIZE, CLIP = 1e-3, 512, 20.0

ALL_SETS = ['cc1_test', 'drift_cc2']

raw = {name: np.load(os.path.join(DATA_DIR, f'X_{name}.npy')) for name in ['cc1_train', 'cc1_val'] + ALL_SETS}
labels = {name: np.load(os.path.join(DATA_DIR, f'y_{name}.npy')) for name in ['cc1_train', 'cc1_val'] + ALL_SETS}
data = {name: np.clip(X, -CLIP, CLIP).astype(np.float32) for name, X in raw.items()}
INPUT_DIM = data['cc1_train'].shape[1]

X_train_t = torch.from_numpy(data['cc1_train'])
X_val_t   = torch.from_numpy(data['cc1_val'])
print(f'INPUT_DIM={INPUT_DIM}  NEW_SEEDS={NEW_SEEDS}  (+ existing seed 42)')

INPUT_DIM=26  NEW_SEEDS=[7, 123, 2024, 999]  (+ existing seed 42)


## VAE class + training loop (identical to `train_vae.ipynb`, KL bug already fixed)

In [3]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Linear(hidden1, hidden2), nn.ReLU())
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden2), nn.ReLU(), nn.Linear(hidden2, hidden1), nn.ReLU(), nn.Linear(hidden1, input_dim))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    def encode(self, x):
        h = self.encoder(x); return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)
    def decode(self, z): return self.decoder(z)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar); return mu + torch.randn_like(std) * std
    def forward(self, x):
        mu, logvar = self.encode(x); z = self.reparameterize(mu, logvar); return self.decode(z), mu, logvar
    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval(); mu, _ = self.encode(x); return ((self.decode(mu) - x) ** 2).mean(dim=1)

def vae_loss(recon, x, mu, logvar, beta):
    recon_loss = nn.functional.mse_loss(recon, x, reduction='mean')
    kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
    return recon_loss + beta * kl, recon_loss, kl

def train_one_seed(seed, max_epochs=FULL_MAX_EPOCHS, patience=FULL_PATIENCE):
    torch.manual_seed(seed); np.random.seed(seed)
    model = VAE(INPUT_DIM, HIDDEN1, HIDDEN2, LATENT_DIM).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loader = DataLoader(TensorDataset(X_train_t), batch_size=BATCH_SIZE, shuffle=True)
    best_val, best_state, patience_ctr = float('inf'), None, 0

    for epoch in range(max_epochs):
        beta = min(1.0, (epoch + 1) / WARMUP_EPOCHS) * BETA_MAX
        model.train()
        for (xb,) in loader:
            opt.zero_grad()
            recon, mu, logvar = model(xb)
            loss, _, _ = vae_loss(recon, xb, mu, logvar, beta)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            recon, mu, logvar = model(X_val_t)
            vloss, _, _ = vae_loss(recon, X_val_t, mu, logvar, beta)
        if vloss.item() < best_val - 1e-6:
            best_val, best_state, patience_ctr = vloss.item(), {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_val, epoch + 1

print('Training function defined.')

Training function defined.


## Evaluate one model: deployed F1 (leak-free threshold) + oracle-best F1, per set

In [4]:
def evaluate_model(model):
    with torch.no_grad():
        mse_val = model.anomaly_score(X_val_t).numpy()
    val_p99 = float(np.percentile(mse_val, 99))

    out = {}
    for name in ALL_SETS:
        X_t = torch.from_numpy(data[name])
        y_true = labels[name]
        with torch.no_grad():
            mse = model.anomaly_score(X_t).numpy()
        auc_roc = roc_auc_score(y_true, mse)
        auc_pr = average_precision_score(y_true, mse)
        pred = (mse > val_p99).astype(int)
        deployed_f1 = f1_score(y_true, pred, zero_division=0)
        deployed_p = precision_score(y_true, pred, zero_division=0)
        deployed_r = recall_score(y_true, pred, zero_division=0)

        p, r, _ = precision_recall_curve(y_true, mse)
        f1s = 2 * p * r / (p + r + 1e-12)
        i = np.argmax(f1s)
        out[name] = {
            'auc_roc': auc_roc, 'auc_pr': auc_pr,
            'deployed_f1': deployed_f1, 'deployed_precision': deployed_p, 'deployed_recall': deployed_r,
            'oracle_f1': float(f1s[i]), 'oracle_precision': float(p[i]), 'oracle_recall': float(r[i]),
        }
    return out, val_p99

print('Evaluation function defined.')

Evaluation function defined.


## Step 1 — Seed 42: load the existing, already-trained model (not retrained)

In [5]:
meta = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_meta.pkl'), 'rb'))
model_42 = VAE(meta['input_dim'], meta['hidden1'], meta['hidden2'], meta['latent_dim'])
model_42.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'vae_cc1.pt'), map_location='cpu'))
model_42.eval()

with torch.no_grad():
    recon, mu, logvar = model_42(X_val_t)
    vloss_42, _, _ = vae_loss(recon, X_val_t, mu, logvar, BETA_MAX)

seed_results = {}
eval_42, val_p99_42 = evaluate_model(model_42)
seed_results[42] = {'val_loss': vloss_42.item(), 'eval': eval_42, 'epochs': None}
print(f'seed=42 (existing model): val_loss={vloss_42.item():.4f}')
print(f'  cc1_test  : oracle_f1={eval_42["cc1_test"]["oracle_f1"]:.4f}  deployed_f1={eval_42["cc1_test"]["deployed_f1"]:.4f}')
print(f'  drift_cc2 : oracle_f1={eval_42["drift_cc2"]["oracle_f1"]:.4f}  deployed_f1={eval_42["drift_cc2"]["deployed_f1"]:.4f}')

seed=42 (existing model): val_loss=0.2324
  cc1_test  : oracle_f1=0.6857  deployed_f1=0.6175
  drift_cc2 : oracle_f1=0.5147  deployed_f1=0.2757


## Step 2 — Train 4 new seeds, evaluate each the same way

In [6]:
for seed in NEW_SEEDS:
    print(f'--- seed={seed} ---')
    t0 = time.time()
    model, best_val, n_epochs = train_one_seed(seed)
    elapsed = time.time() - t0
    eval_result, val_p99 = evaluate_model(model)
    seed_results[seed] = {'val_loss': best_val, 'eval': eval_result, 'epochs': n_epochs}
    print(f'  trained {n_epochs} epochs, val_loss={best_val:.4f}  ({elapsed:.0f}s)')
    print(f'  cc1_test  : oracle_f1={eval_result["cc1_test"]["oracle_f1"]:.4f}  deployed_f1={eval_result["cc1_test"]["deployed_f1"]:.4f}')
    print(f'  drift_cc2 : oracle_f1={eval_result["drift_cc2"]["oracle_f1"]:.4f}  deployed_f1={eval_result["drift_cc2"]["deployed_f1"]:.4f}')
    print()

--- seed=7 ---
  trained 255 epochs, val_loss=0.2503  (652s)
  cc1_test  : oracle_f1=0.6260  deployed_f1=0.4573
  drift_cc2 : oracle_f1=0.5509  deployed_f1=0.1619

--- seed=123 ---
  trained 300 epochs, val_loss=0.2421  (754s)
  cc1_test  : oracle_f1=0.6818  deployed_f1=0.5594
  drift_cc2 : oracle_f1=0.5578  deployed_f1=0.1765

--- seed=2024 ---
  trained 275 epochs, val_loss=0.2401  (665s)
  cc1_test  : oracle_f1=0.6568  deployed_f1=0.5812
  drift_cc2 : oracle_f1=0.3346  deployed_f1=0.1802

--- seed=999 ---
  trained 186 epochs, val_loss=0.2467  (450s)
  cc1_test  : oracle_f1=0.6986  deployed_f1=0.6092
  drift_cc2 : oracle_f1=0.5369  deployed_f1=0.2027



## Step 3 — Full comparison table, sorted by validation loss (the leak-free selection criterion)

In [7]:
seeds_sorted = sorted(seed_results.keys(), key=lambda s: seed_results[s]['val_loss'])
print(f'{"seed":>6s} {"val_loss":>9s} {"cc1_test oracle_f1":>19s} {"cc1_test deployed_f1":>21s} {"drift_cc2 oracle_f1":>20s} {"drift_cc2 deployed_f1":>22s}')
for seed in seeds_sorted:
    r = seed_results[seed]
    marker = '  <- lowest val_loss (leak-free pick)' if seed == seeds_sorted[0] else ''
    print(f'{seed:>6d} {r["val_loss"]:9.4f} {r["eval"]["cc1_test"]["oracle_f1"]:19.4f} {r["eval"]["cc1_test"]["deployed_f1"]:21.4f} '
          f'{r["eval"]["drift_cc2"]["oracle_f1"]:20.4f} {r["eval"]["drift_cc2"]["deployed_f1"]:22.4f}{marker}')

best_by_val_loss = seeds_sorted[0]
best_by_test_oracle = max(seed_results, key=lambda s: seed_results[s]['eval']['cc1_test']['oracle_f1'])
print(f'\nLeak-free pick (lowest val_loss): seed={best_by_val_loss}')
print(f'Best test-oracle-F1 pick (NOT leak-free — chose using cc1_test): seed={best_by_test_oracle}')
if best_by_val_loss != best_by_test_oracle:
    print('  -> These differ. Reporting the val_loss pick as primary is the fully clean choice.')
else:
    print('  -> These agree — the leak-free pick also happens to be the best on test. Clean result either way.')

  seed  val_loss  cc1_test oracle_f1  cc1_test deployed_f1  drift_cc2 oracle_f1  drift_cc2 deployed_f1
    42    0.2324              0.6857                0.6175               0.5147                 0.2757  <- lowest val_loss (leak-free pick)
  2024    0.2401              0.6568                0.5812               0.3346                 0.1802
   123    0.2421              0.6818                0.5594               0.5578                 0.1765
   999    0.2467              0.6986                0.6092               0.5369                 0.2027
     7    0.2503              0.6260                0.4573               0.5509                 0.1619

Leak-free pick (lowest val_loss): seed=42
Best test-oracle-F1 pick (NOT leak-free — chose using cc1_test): seed=999
  -> These differ. Reporting the val_loss pick as primary is the fully clean choice.


## Step 4 — Does any seed cross 0.7?

In [8]:
print('cc1_test oracle F1 across all seeds tested:')
for seed in seeds_sorted:
    v = seed_results[seed]['eval']['cc1_test']['oracle_f1']
    flag = '  *** EXCEEDS 0.7 ***' if v > 0.7 else ''
    print(f'  seed={seed:>5d}: oracle_f1={v:.4f}{flag}')
best_ceiling = max(seed_results[s]['eval']['cc1_test']['oracle_f1'] for s in seed_results)
print(f'\nBest oracle ceiling found across {len(seed_results)} seeds: {best_ceiling:.4f}')
print('(Previous single-seed ceiling was 0.686 — this either confirms that\'s close to a real limit, or finds something higher.)')

cc1_test oracle F1 across all seeds tested:
  seed=   42: oracle_f1=0.6857
  seed= 2024: oracle_f1=0.6568
  seed=  123: oracle_f1=0.6818
  seed=  999: oracle_f1=0.6986
  seed=    7: oracle_f1=0.6260

Best oracle ceiling found across 5 seeds: 0.6986
(Previous single-seed ceiling was 0.686 — this either confirms that's close to a real limit, or finds something higher.)


## Step 5 — Save results

In [9]:
save_results = {
    'fixed_hyperparams': {'hidden1': HIDDEN1, 'hidden2': HIDDEN2, 'latent_dim': LATENT_DIM, 'beta_max': BETA_MAX},
    'seeds': {seed: {'val_loss': r['val_loss'], 'epochs': r['epochs'], 'eval': r['eval']} for seed, r in seed_results.items()},
    'best_by_val_loss': best_by_val_loss,
    'best_by_test_oracle': best_by_test_oracle,
}
out_path = os.path.join(MODEL_DIR, 'multi_seed_oracle_ceiling.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {out_path}')

Saved -> c:\Users\jthar\Documents\Claude\Projects\module3\models\multi_seed_oracle_ceiling.pkl


## How to read this

- If the best oracle ceiling across all 5 seeds is still close to 0.686, that's
  strong evidence 0.7 isn't reachable via random-seed variation alone with this
  architecture/feature set — a real, informative negative result, not a failed
  experiment.
- If a seed's ceiling clearly exceeds 0.7, check **which selection rule** found
  it: if it's the same seed picked by lowest `val_loss`, that's a fully
  leak-free, reportable improvement. If only the "best on cc1_test" rule finds
  it, say so explicitly if you report that number — it's a legitimate result
  from a small (5-way) comparison, just not as clean as a val-loss-based pick.